<a href="https://colab.research.google.com/github/kalleahi/beneish-mscore-sp500-colab/blob/main/Beneish_SP500_MScore_Colab_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 📗 Beneish M‑Score Screener for the S&P 500 — **v4 (Student Edition)**
This notebook computes the **Beneish M‑Score** for S&P 500 companies to *screen* for potential earnings manipulation risk.

**NB!** To interact with this notebook (run code, make changes, or save your work), please go to `File > Save a copy in Drive` to create your own editable version. You will not be able to run code or make permanent changes to this shared version.

**What’s inside**
1. 📦 Install dependencies (Colab-friendly)
2. 📋 Pull S&P 500 constituents (from Wikipedia)
3. 🧮 Fetch financials via `yfinance` (annual ➜ quarterly fallback)
4. 🔎 Compute the 8 Beneish inputs (DSRI, GMI, AQI, SGI, DEPI, SGAI, LVGI, TATA) and the **M‑Score**
5. 🧰 Robust label matching using fuzzy search (handles "SG&A", "Selling General And Administrative", etc.)
6. 🧾 **Data lineage** columns: which exact labels were matched for each field
7. 📤 Export **CSV** and **Excel** (with red/green conditional formatting and a “Flagged” filter)
8. ⚠️ Caveats and interpretation tips

> **Reminder:** The Beneish M‑Score is a **screening tool** — **not** proof of manipulation. Always inspect filings and context.



## 🎯 Learning goals
By running this notebook, you will be able to:
- Explain the intuition behind each Beneish ratio
- Reproduce a systematic screen on a large universe (S&P 500)
- Interpret results with appropriate caution
- Export results for further analysis (CSV / Excel)



## 1) Install packages
We use:
- `yfinance` for financial statements
- `pandas`, `numpy` for data wrangling
- `rapidfuzz` for robust label matching
- `tqdm` for progress bars
- `openpyxl` / `xlsxwriter` for Excel export (conditional formatting)


In [ ]:
!pip -q install yfinance bs4 lxml pandas numpy requests tqdm rapidfuzz openpyxl XlsxWriter rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.5 MB/s eta 0:00:00



## 2) Imports and global configuration
- `TEST_LIMIT` lets you limit the number of tickers for a quick test run.
- `BENEISH_THRESHOLD` is set to **−1.78** (classic cutoff).

In [ ]:

import pandas as pd
import numpy as np
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from tqdm import tqdm
from rapidfuzz import process, fuzz

pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 200)

# ▶ Set to a small integer (e.g., 30) for quick tests; None for full S&P 500
TEST_LIMIT = None

# Output filenames
OUTPUT_CSV = "sp500_beneish_scores_v4.csv"
OUTPUT_XLSX = "sp500_beneish_scores_v4.xlsx"

# Beneish threshold (screen)
BENEISH_THRESHOLD = -1.78



## 3) Pull the current S&P 500 list
We scrape Wikipedia’s *List of S&P 500 companies* and standardize tickers (e.g., `BRK.B` ➜ `BRK-B` for yfinance).

In [ ]:
def fetch_sp500_from_wikipedia() -> pd.DataFrame:
    """Fetch S&P 500 constituents from Wikipedia, including GICS sector metadata."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    r = requests.get(url, timeout=30, headers=headers)
    r.raise_for_status()

    tables = pd.read_html(r.text)
    for tbl in tables:
        # The main constituents table contains both Symbol and GICS Sector columns
        if ("Symbol" in tbl.columns) and ("GICS Sector" in tbl.columns):
            df = tbl.copy()
            df.rename(columns={"Symbol": "Ticker", "Security": "Company"}, inplace=True)

            # Wikipedia uses '.' in some tickers (e.g., BRK.B) while Yahoo Finance uses '-'
            df["Ticker"] = df["Ticker"].astype(str).str.replace(".", "-", regex=False).str.strip()

            keep_cols = ["Ticker", "Company", "GICS Sector", "GICS Sub-Industry"]
            keep_cols = [c for c in keep_cols if c in df.columns]
            return df[keep_cols]

    raise RuntimeError("Could not locate S&P 500 table with GICS columns on Wikipedia.")


sp500 = fetch_sp500_from_wikipedia()
if TEST_LIMIT:
    sp500 = sp500.head(TEST_LIMIT).copy()

# Exclude Financials for Beneish M-score calculation (financial statements are not comparable)
sp500_non_fin = sp500[sp500["GICS Sector"].ne("Financials")].copy()

print(f"Total tickers: {len(sp500)} | Non-Financials: {len(sp500_non_fin)}")
sp500_non_fin.head(10)


/tmp/ipython-input-3963034121.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)


Total tickers: 503 | Non-Financials: 427


,Ticker,Company,GICS Sector,GICS Sub-Industry
0,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,A. O. Smith,Industrials,Building Products
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,Accenture,Information Technology,IT Consulting & Other Services
5,ADBE,Adobe Inc.,Information Technology,Application Software
6,AMD,Advanced Micro Devices,Information Technology,Semiconductors
7,AES,AES Corporation,Utilities,Independent Power Producers & Energy Traders
9,A,Agilent Technologies,Health Care,Life Sciences Tools & Services
10,APD,Air Products,Materials,Industrial Gases



## 4) About the data & missing values
Financial statement labels can vary across companies. To make this robust, we:
- Try **multiple label variants** (e.g., *Revenue*, *Total Revenue*, *Sales*).
- Use **fuzzy matching** (RapidFuzz) if exact labels are missing.
- Fall back to **quarterly** statements when annual ones lack two periods.
- If SG&A is missing, we approximate: **SG&A ≈ Operating Expenses − R&D**.

We also record **which label matched** for each field in `Match_*` columns (data lineage).


In [ ]:

from typing import Optional, Tuple, Dict

# Candidate label variants
KEYS = {
    'sales': ["Total Revenue","TotalRevenue","Revenues","Revenue","Sales"],
    'cogs': ["Cost Of Revenue","CostOfRevenue","Cost of Goods Sold","COGS"],
    'sga': ["Selling General Administrative","SellingGeneralAdministrative","Selling General And Administrative","SG&A Expense","SG&A"],
    'ni': ["Net Income","NetIncome"],
    'ar': ["Net Receivables","NetReceivables","Accounts Receivable","Trade And Other Receivables","AccountsReceivable"],
    'ta': ["Total Assets","TotalAssets","Assets"],
    'ca': ["Total Current Assets","TotalCurrentAssets","Current Assets","CurrentAssets"],
    'ppe': ["Property Plant And Equipment Net","PropertyPlantAndEquipmentNet","Property, Plant & Equipment Net","Net PPE","NetPPE","Net Property Plant & Equipment"],
    'depr': ["Depreciation","Depreciation & Amortization","DepreciationAmortization","Depreciation And Amortization", "Reconciled Depreciation"],
    'cfo': ["Total Cash From Operating Activities","TotalCashFromOperatingActivities","Operating Cash Flow","Net Cash Provided By Operating Activities"],
    'tot_debt': ["Total Debt","TotalDebt","Short Long Term Debt Total","ShortLongTermDebtTotal"],
    'st_debt': ["Short Long Term Debt","ShortLongTermDebt","Short Term Debt","ShortTermDebt","Current Portion Of Long Term Debt"],
    'lt_debt': ["Long Term Debt","LongTermDebt"],
    'opex': ["Total Operating Expenses","Operating Expense","Operating Expenses","OperatingExpenses"],
    'rnd': ["Research Development","Research And Development","R&D","ResearchAndDevelopment"]
}

def fuzzy_get_with_lineage(series: pd.Series, keys: list, scorer=fuzz.WRatio, score_cutoff=88) -> Tuple[Optional[float], Optional[str]]:
    """Return (value, matched_label). Try exact keys first, then fuzzy-match by RapidFuzz."""
    # exact match
    for k in keys:
        if k in series.index:
            return series[k], k
    # fuzzy match
    choices = list(series.index)
    for k in keys:
        match = process.extractOne(k, choices, scorer=scorer, score_cutoff=score_cutoff)
        if match:
            return series[match[0]], match[0]
    return None, None

def extract_pair(df: pd.DataFrame, quarterly_df: pd.DataFrame) -> Tuple[Optional[pd.Series], Optional[pd.Series]]:
    """Return (series_t, series_t1) trying annual df first, else quarterly df."""
    def two_cols(dfi):
        if dfi is None or dfi.empty or len(dfi.columns) < 2:
            return None
        return dfi.iloc[:,0], dfi.iloc[:,1]
    ann = two_cols(df)
    if ann:
        return ann
    qtr = two_cols(quarterly_df)
    return qtr

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


In [ ]:

def compute_beneish_for_ticker(ticker: str, company_hint: Optional[str]=None) -> Dict:
    info = { 'Ticker': ticker, 'Company': company_hint, 'Year_t': None, 'Year_t_1': None,
             # inputs
             'Sales_t': np.nan,'Sales_t_1': np.nan,'COGS_t': np.nan,'COGS_t_1': np.nan,
             'Receivables_t': np.nan,'Receivables_t_1': np.nan,'TotalAssets_t': np.nan,'TotalAssets_t_1': np.nan,
             'CurrentAssets_t': np.nan,'CurrentAssets_t_1': np.nan,'PPE_t': np.nan,'PPE_t_1': np.nan,
             'Dep_t': np.nan,'Dep_t_1': np.nan,'SGA_t': np.nan,'SGA_t_1': np.nan,
             'TotalDebt_t': np.nan,'TotalDebt_t_1': np.nan,'CFO_t': np.nan,'NI_t': np.nan,
             # ratios
             'DSRI': np.nan,'GMI': np.nan,'AQI': np.nan,'SGI': np.nan,'DEPI': np.nan,'SGAI': np.nan,'LVGI': np.nan,'TATA': np.nan,
             'MScore': np.nan,'is_suspicious': np.nan,'Note': None }
    # data lineage (which labels matched)
    lineage = {f"Match_{k}": None for k in ['Sales_t','Sales_t_1','COGS_t','COGS_t_1','Receivables_t','Receivables_t_1',
                                            'TotalAssets_t','TotalAssets_t_1','CurrentAssets_t','CurrentAssets_t_1',
                                            'PPE_t','PPE_t_1','Dep_t','Dep_t_1','SGA_t','SGA_t_1','TotalDebt_t','TotalDebt_t_1','CFO_t','NI_t']}
    try:
        tk = yf.Ticker(ticker)
        try:
            if not info['Company']:
                info['Company'] = tk.info.get('longName')
        except Exception:
            pass

        bs_ann = tk.balance_sheet
        is_ann = tk.financials
        cf_ann = tk.cashflow
        bs_q = tk.quarterly_balance_sheet
        is_q = tk.quarterly_financials
        cf_q = tk.quarterly_cashflow

        bs_t, bs_t1 = extract_pair(bs_ann, bs_q)
        is_t, is_t1 = extract_pair(is_ann, is_q)
        cf_t, cf_t1 = extract_pair(cf_ann, cf_q)

        if bs_t is None or bs_t1 is None or is_t is None or is_t1 is None or cf_t is None or cf_t1 is None:
            info['Note'] = "Missing two periods in at least one statement (after quarterly fallback)."

        # period labels (best effort)
        def get_year_from_series(s):
            try:
                return str(s.name.year)
            except Exception:
                try:
                    return str(s.name)
                except Exception:
                    return None
        if isinstance(bs_t, pd.Series):
            info['Year_t'] = get_year_from_series(bs_t)
        if isinstance(bs_t1, pd.Series):
            info['Year_t_1'] = get_year_from_series(bs_t1)

        # Income statement (t, t-1)
        val, lab = fuzzy_get_with_lineage(is_t, KEYS['sales']) if is_t is not None else (None,None)
        info['Sales_t'], lineage['Match_Sales_t'] = safe_float(val), lab
        val, lab = fuzzy_get_with_lineage(is_t1, KEYS['sales']) if is_t1 is not None else (None,None)
        info['Sales_t_1'], lineage['Match_Sales_t_1'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(is_t, KEYS['cogs']) if is_t is not None else (None,None)
        info['COGS_t'], lineage['Match_COGS_t'] = safe_float(val), lab
        val, lab = fuzzy_get_with_lineage(is_t1, KEYS['cogs']) if is_t1 is not None else (None,None)
        info['COGS_t_1'], lineage['Match_COGS_t_1'] = safe_float(val), lab

        # SG&A; fallback via OPEX - R&D
        val, lab = fuzzy_get_with_lineage(is_t, KEYS['sga']) if is_t is not None else (None,None)
        if val is None and is_t is not None:
            opex, lab_opex = fuzzy_get_with_lineage(is_t, KEYS['opex'])
            rnd, lab_rnd = fuzzy_get_with_lineage(is_t, KEYS['rnd'])
            if opex is not None:
                val = (opex - (rnd or 0))
                lab = f"{lab_opex} - ({lab_rnd or '0'})"
        info['SGA_t'], lineage['Match_SGA_t'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(is_t1, KEYS['sga']) if is_t1 is not None else (None,None)
        if val is None and is_t1 is not None:
            opex, lab_opex = fuzzy_get_with_lineage(is_t1, KEYS['opex'])
            rnd, lab_rnd = fuzzy_get_with_lineage(is_t1, KEYS['rnd'])
            if opex is not None:
                val = (opex - (rnd or 0))
                lab = f"{lab_opex} - ({lab_rnd or '0'})"
        info['SGA_t_1'], lineage['Match_SGA_t_1'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(is_t, KEYS['ni']) if is_t is not None else (None,None)
        info['NI_t'], lineage['Match_NI_t'] = safe_float(val), lab

        # Balance sheet
        val, lab = fuzzy_get_with_lineage(bs_t, KEYS['ar']) if bs_t is not None else (None,None)
        info['Receivables_t'], lineage['Match_Receivables_t'] = safe_float(val), lab
        val, lab = fuzzy_get_with_lineage(bs_t1, KEYS['ar']) if bs_t1 is not None else (None,None)
        info['Receivables_t_1'], lineage['Match_Receivables_t_1'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(bs_t, KEYS['ta']) if bs_t is not None else (None,None)
        info['TotalAssets_t'], lineage['Match_TotalAssets_t'] = safe_float(val), lab
        val, lab = fuzzy_get_with_lineage(bs_t1, KEYS['ta']) if bs_t1 is not None else (None,None)
        info['TotalAssets_t_1'], lineage['Match_TotalAssets_t_1'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(bs_t, KEYS['ca']) if bs_t is not None else (None,None)
        info['CurrentAssets_t'], lineage['Match_CurrentAssets_t'] = safe_float(val), lab
        val, lab = fuzzy_get_with_lineage(bs_t1, KEYS['ca']) if bs_t1 is not None else (None,None)
        info['CurrentAssets_t_1'], lineage['Match_CurrentAssets_t_1'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(bs_t, KEYS['ppe']) if bs_t is not None else (None,None)
        info['PPE_t'], lineage['Match_PPE_t'] = safe_float(val), lab
        val, lab = fuzzy_get_with_lineage(bs_t1, KEYS['ppe']) if bs_t1 is not None else (None,None)
        info['PPE_t_1'], lineage['Match_PPE_t_1'] = safe_float(val), lab

        # Debt
        val, lab = fuzzy_get_with_lineage(bs_t, KEYS['tot_debt']) if bs_t is not None else (None,None)
        if val is None and bs_t is not None:
            st, lab_st = fuzzy_get_with_lineage(bs_t, KEYS['st_debt'])
            lt, lab_lt = fuzzy_get_with_lineage(bs_t, KEYS['lt_debt'])
            if st is not None or lt is not None:
                val = (st or 0) + (lt or 0)
                lab = f"{lab_st or 0} + {lab_lt or 0}"
        info['TotalDebt_t'], lineage['Match_TotalDebt_t'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(bs_t1, KEYS['tot_debt']) if bs_t1 is not None else (None,None)
        if val is None and bs_t1 is not None:
            st1, lab_st1 = fuzzy_get_with_lineage(bs_t1, KEYS['st_debt'])
            lt1, lab_lt1 = fuzzy_get_with_lineage(bs_t1, KEYS['lt_debt'])
            if st1 is not None or lt1 is not None:
                val = (st1 or 0) + (lt1 or 0)
                lab = f"{lab_st1 or 0} + {lab_lt1 or 0}"
        info['TotalDebt_t_1'], lineage['Match_TotalDebt_t_1'] = safe_float(val), lab

        # Cash flow
        val, lab = fuzzy_get_with_lineage(cf_t, KEYS['depr']) if cf_t is not None else (None,None)
        info['Dep_t'], lineage['Match_Dep_t'] = safe_float(val), lab
        val, lab = fuzzy_get_with_lineage(cf_t1, KEYS['depr']) if cf_t1 is not None else (None,None)
        info['Dep_t_1'], lineage['Match_Dep_t_1'] = safe_float(val), lab

        val, lab = fuzzy_get_with_lineage(cf_t, KEYS['cfo']) if cf_t is not None else (None,None)
        info['CFO_t'], lineage['Match_CFO_t'] = safe_float(val), lab

        # Ratios
        def sdiv(a,b):
            return np.nan if (a is None or b is None or np.isnan(a) or np.isnan(b) or b==0) else a/b

        dsri = sdiv(sdiv(info['Receivables_t'], info['Sales_t']), sdiv(info['Receivables_t_1'], info['Sales_t_1']))
        margin_t = sdiv((info['Sales_t']-info['COGS_t']), info['Sales_t'])
        margin_t1 = sdiv((info['Sales_t_1']-info['COGS_t_1']), info['Sales_t_1'])
        gmi = sdiv(margin_t1, margin_t)
        aqi_num_t = 1 - sdiv((info['CurrentAssets_t'] + info['PPE_t']), info['TotalAssets_t'])
        aqi_num_t1 = 1 - sdiv((info['CurrentAssets_t_1'] + info['PPE_t_1']), info['TotalAssets_t_1'])
        aqi = sdiv(aqi_num_t, aqi_num_t1)
        sgi = sdiv(info['Sales_t'], info['Sales_t_1'])
        depi = sdiv(sdiv(info['Dep_t_1'], info['PPE_t_1']), sdiv(info['Dep_t'], info['PPE_t']))
        sgai = sdiv(sdiv(info['SGA_t'], info['Sales_t']), sdiv(info['SGA_t_1'], info['Sales_t_1']))
        lvgi = sdiv(sdiv(info['TotalDebt_t'], info['TotalAssets_t']), sdiv(info['TotalDebt_t_1'], info['TotalAssets_t_1']))
        tata = sdiv((info['NI_t'] - info['CFO_t']), info['TotalAssets_t'])

        info.update({'DSRI': dsri,'GMI': gmi,'AQI': aqi,'SGI': sgi,'DEPI': depi,'SGAI': sgai,'LVGI': lvgi,'TATA': tata})

        # M-Score
        comps = ['DSRI','GMI','AQI','SGI','DEPI','SGAI','TATA','LVGI']
        if any(np.isnan(info[k]) for k in comps):
            info['MScore'] = np.nan
            info['is_suspicious'] = np.nan
        else:
            coeffs = {'const': -4.84,'DSRI': 0.920,'GMI': 0.528,'AQI': 0.404,'SGI': 0.892,'DEPI': 0.115,'SGAI': -0.172,'TATA': 4.679,'LVGI': -0.327}
            info['MScore'] = coeffs['const'] + sum(coeffs[k]*info[k] for k in comps)
            info['is_suspicious'] = bool(info['MScore'] > BENEISH_THRESHOLD)

        # merge lineage into info
        info.update(lineage)
        return info
    except Exception as e:
        info['Note'] = f"Error: {e}"
        info.update(lineage)
        return info



## 5) Run the screen and export results
This will iterate over the S&P 500 list, compute the M‑Score, and then save.
This may take around 10 minutes for full S&P500 list:
- `sp500_beneish_scores_v3.csv`
- `sp500_beneish_scores_v3.xlsx` (with conditional formatting)


In [ ]:

records = []
for _, row in tqdm(sp500_non_fin.iterrows(), total=len(sp500_non_fin)):
    rec = compute_beneish_for_ticker(row['Ticker'], company_hint=row.get('Company'))
    # Attach GICS metadata
    rec['GICS Sector'] = row.get('GICS Sector')
    rec['GICS Sub-Industry'] = row.get('GICS Sub-Industry')
    records.append(rec)

df = pd.DataFrame(records)

front = ['Ticker','Company','GICS Sector','GICS Sub-Industry','Year_t','Year_t_1','MScore','is_suspicious','Note']
ratios = ['DSRI','GMI','AQI','SGI','DEPI','SGAI','LVGI','TATA']
inputs = ['Sales_t','Sales_t_1','COGS_t','COGS_t_1','Receivables_t','Receivables_t_1',
          'TotalAssets_t','TotalAssets_t_1','CurrentAssets_t','CurrentAssets_t_1',
          'PPE_t','PPE_t_1','Dep_t','Dep_t_1','SGA_t','SGA_t_1','TotalDebt_t','TotalDebt_t_1',
          'CFO_t','NI_t']
lineage_cols = [c for c in df.columns if c.startswith('Match_')]

# Keep only columns that exist (robust to minor schema changes)
ordered = [c for c in (front+ratios+inputs+lineage_cols) if c in df.columns]
df = df.reindex(columns=ordered)

# Save CSV
df.to_csv(OUTPUT_CSV, index=False)
print("Saved CSV:", OUTPUT_CSV)

# Save Excel with formatting
with pd.ExcelWriter(OUTPUT_XLSX, engine='xlsxwriter') as writer:
    df.to_excel(writer, index=False, sheet_name='Beneish')
    wb  = writer.book
    ws  = writer.sheets['Beneish']
    # Header freeze
    ws.freeze_panes(1, 0)
    # Autofilter
    ws.autofilter(0, 0, len(df), len(df.columns)-1)
    # Conditional formatting on MScore and is_suspicious (if present)
    if 'MScore' in df.columns and 'is_suspicious' in df.columns:
        mscore_col = df.columns.get_loc('MScore')
        susp_col = df.columns.get_loc('is_suspicious')
        # Color scale for MScore (higher = more suspicious)
        ws.conditional_format(1, mscore_col, len(df), mscore_col, {'type':'3_color_scale'})
        # Boolean flag formatting
        ws.conditional_format(1, susp_col, len(df), susp_col,
                              {'type':'cell',
                               'criteria':'==',
                               'value': True,
                               'format': wb.add_format({'font_color':'#9C0006','bold':True})})
print("Saved Excel:", OUTPUT_XLSX)

# Show top 25 highest M-Scores
show_cols = [c for c in ['Ticker','Company','GICS Sector','MScore','is_suspicious'] if c in df.columns]
df[df['MScore'].notna()].sort_values('MScore', ascending=False).head(25)[show_cols]


100%|██████████| 427/427 [09:05<00:00,  1.28s/it]


Saved CSV: sp500_beneish_scores_v4.csv
Saved Excel: sp500_beneish_scores_v4.xlsx


,Ticker,Company,GICS Sector,MScore,is_suspicious
146,EQT,EQT Corporation,Energy,2.298449,True
12,ALB,Albemarle Corporation,Materials,1.734330,True
155,EXE,Expand Energy,Energy,0.624928,True
269,MPWR,Monolithic Power Systems,Information Technology,-0.035159,True
305,PAYC,Paycom,Industrials,-1.320589,True
354,LUV,Southwest Airlines,Industrials,-1.644822,True
21,AMCR,Amcor,Materials,-1.653365,True
165,FSLR,First Solar,Information Technology,-1.760338,True
44,AXON,Axon Enterprise,Industrials,-1.870432,False
195,HD,Home Depot (The),Consumer Discretionary,-1.884651,False



## 6) Download your files
In Colab, run the cell below to download the CSV or Excel directly to your computer.


In [ ]:

from google.colab import files
print("Uncomment one or both lines and run to download:")
# files.download('sp500_beneish_scores_v4.csv')
# files.download('sp500_beneish_scores_v4.xlsx')


Uncomment one or both lines and run to download:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## 7) Interpreting the Beneish M‑Score
- **Cutoff:** A common threshold is **−1.78**. Values **> −1.78** (less negative or positive) suggest *possible* manipulation.
- **Components:**
  - **DSRI** (Days’ Sales in Receivables Index): looking for aggressive revenue recognition.
  - **GMI** (Gross Margin Index): deterioration in gross margin may pressure manipulation.
  - **AQI** (Asset Quality Index): growth in intangible/less reliable assets.
  - **SGI** (Sales Growth Index): rapid growth sometimes correlates with manipulation.
  - **DEPI** (Depreciation Index): changes in depreciation can signal policy shifts.
  - **SGAI** (SG&A Index): falling SG&A/Sales ratio may be a red flag.
  - **LVGI** (Leverage Index): higher leverage can create incentives.
  - **TATA** (Total Accruals to Total Assets): higher accruals can be a warning.
- **Not a proof:** Use as a **screen**. Always inspect MD&A, footnotes, revenue recognition policies, one-offs, and quality of cash flows.



## 8) Caveats & potential improvements
- **Data heterogeneity:** Different label names and statement structures remain a challenge despite fuzzy matching.
- **Mixing annual & quarterly:** When annual data is missing, we fall back to **two most recent quarters** — period-end mismatch is possible.
- **DEPI proxy:** We use `(Depreciation / PPE)` as a proxy; the original paper uses accumulated depreciation.
- **Next steps:** For production use, consider pulling directly from the **SEC API** (10‑K / 10‑Q XBRL tags) for stricter line-item definitions.



## 9) Debug a single ticker (optional)
Use this to see raw inputs, matches, and the computed M‑Score for a specific company.


In [ ]:

ticker_to_debug = "AAPL"  # change as needed
rec = compute_beneish_for_ticker(ticker_to_debug)
pd.Series(rec)


,0
Ticker,AAPL
Company,Apple Inc.
Year_t,2025
Year_t_1,2024
Sales_t,416161000000.0
Sales_t_1,391035000000.0
COGS_t,220960000000.0
COGS_t_1,210352000000.0
Receivables_t,39777000000.0
Receivables_t_1,33410000000.0
